In [1]:
import transformers
print(transformers.__version__)

C:\Users\epuerta\AppData\Local\anaconda3\envs\python-nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4.57.1


In [2]:
# conda install -c huggingface -c conda-forge datasets

In [3]:
import os
import sys
PATH = os.getcwd()
DIR_DATA = PATH + '{0}data{0}'.format(os.sep)
sys.path.append(PATH) if PATH not in list(sys.path) else None

In [4]:
os.makedirs("./logs", exist_ok=True)
os.makedirs("./results", exist_ok=True)
os.environ["WANDB_DISABLED"] = "true"

# Paso 1: Importación de librerías

In [5]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import numpy as np
 

ModuleNotFoundError: No module named 'datasets'

# Paso 2: Carga y limpieza del dataset

In [ ]:
df = pd.read_csv(DIR_DATA + "dataset_sentimientos_500.csv")

In [ ]:
df.columns = df.columns.str.strip()

In [ ]:
df = df[['Reseña', 'Sentimiento']].dropna()
df['Sentimiento'] = df['Sentimiento'].map({'Positiva': 1, 'Negativa': 0})

In [ ]:
df

# Paso 3: Separación en entrenamiento y prueba

In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
df['Reseña'].tolist(), df['Sentimiento'].tolist(), test_size=0.2, random_state=42)

# Paso 4: Tokenización

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased') 

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128) 

test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128) 

# Paso 5: Creación de los datasets formales 

In [ ]:
train_dataset = Dataset.from_dict({ 

    'input_ids': train_encodings['input_ids'], 

    'attention_mask': train_encodings['attention_mask'], 

    'labels': train_labels 

}) 

test_dataset = Dataset.from_dict({ 

    'input_ids': test_encodings['input_ids'], 

    'attention_mask': test_encodings['attention_mask'], 

    'labels': test_labels 

})

# Paso 6: Definición de métricas 

In [ ]:
def compute_metrics(eval_pred): 

    logits, labels = eval_pred 

    preds = np.argmax(logits, axis=-1) 

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary') 

    acc = accuracy_score(labels, preds) 

    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall} 

# Paso 7: Carga del modelo preentrenado

In [ ]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2) 

# Paso 8: Configuración del entrenamiento 

In [ ]:
Training_args = TrainingArguments(
    output_dir="./results",
    logging_dir="./logs",
    logging_steps=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    evaluation_strategy="steps",  # 👈 evaluar al final de cada época
    save_strategy="steps",         # 👈 guardar al final de cada época
    report_to="none"               # 👈 evita el login de wandb
)

# Paso 9: Entrenamiento del modelo 

In [ ]:
trainer = Trainer( 

    model=model, 

    args=training_args, 

    train_dataset=train_dataset, 

    eval_dataset=test_dataset, 

    compute_metrics=compute_metrics 

) 

trainer.train() 

# Paso 10: Evaluación final del modelo

In [ ]:
results = trainer.evaluate() 

print("Resultados:", results) 